# Model speed-up with factor `s`: origin and application

## Goal
Estimate stratospheric perturbation lifetimes (`tau`) across altitude and compute a first-year speed-up factor (`s`) for model experiments.

## Inputs
- Lifetime reference points at selected altitudes from previous simulations
- Linear and quadratic interpolation/extrapolation

## Outputs
- Estimated lifetime profile (`tau`) between 15 and 43 km
- Example `s` values at 30 km and 38 km


## 1. Analytical background

For a quasi-steady trace-gas perturbation mass `X` [kg], annual emission `E` [kg/year], and perturbation lifetime `tau` [years]:

$$
0 = rac{dX}{dt} = E - rac{1}{	au}X
$$

which gives:

$$
X = 	au E
$$


To derive the speed-up factor `s`, use the stratospheric perturbation mass `Y` with accelerated year-1 emission `sE`:

$$
rac{dY}{dt} = sE - rac{1}{	au}Y
$$

with boundary conditions `Y(0) = 0` and `Y(1) = tau E`.
The exact relation is:

$$
s = rac{1}{1 - e^{-1/	au}}
$$

For practical use in this notebook, we apply the approximation:

$$
s pprox 	au + 0.5
$$

This approximation is typically used for `tau` values around 2 to 5 years.


## 2. Example application

### Import modules


In [ ]:
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt


### Input data

Reference perturbation lifetimes from previous studies:
- lower-altitude points: Grewe & Stenke (2008)
- upper-altitude points: Pletzer et al. (2022, under review)


In [ ]:
altitude_km = np.array([16, 20, 26, 35], dtype=float)
tau_years = np.array([8 / 12, 2.0, 3.5, 4.6], dtype=float)


### Helper functions


In [ ]:
def build_tau_estimators(altitudes_km, lifetimes_years):
    """Return linear and quadratic lifetime estimators with extrapolation."""
    linear = interp1d(altitudes_km, lifetimes_years, kind='linear', fill_value='extrapolate')
    quadratic = interp1d(altitudes_km, lifetimes_years, kind='quadratic', fill_value='extrapolate')
    return linear, quadratic


def average_tau(linear_estimator, quadratic_estimator, query_altitudes_km):
    """Average linear and quadratic tau estimates at the requested altitudes."""
    tau_linear = linear_estimator(query_altitudes_km)
    tau_quadratic = quadratic_estimator(query_altitudes_km)
    tau_average = 0.5 * (tau_linear + tau_quadratic)
    return tau_average, tau_linear, tau_quadratic


def approximate_factor_s(tau_values_years):
    """Taylor-based approximation used in this notebook."""
    return tau_values_years + 0.5


### Compute lifetime estimates on an altitude grid


In [ ]:
linear_tau, quadratic_tau = build_tau_estimators(altitude_km, tau_years)

altitude_grid_km = np.arange(15, 44, 2, dtype=float)
tau_avg_grid, tau_linear_grid, tau_quadratic_grid = average_tau(
    linear_tau,
    quadratic_tau,
    altitude_grid_km,
)


### Evaluate key altitudes (30 km and 38 km)


In [ ]:
target_altitudes_km = np.array([30.0, 38.0], dtype=float)
tau_avg_target, tau_linear_target, tau_quadratic_target = average_tau(
    linear_tau,
    quadratic_tau,
    target_altitudes_km,
)


### Visualize results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(altitude_km, tau_years, s=60, color='tab:blue', label='tau input values')
ax.plot(altitude_grid_km, tau_linear_grid, '-.', color='tab:red', lw=2.5, label='tau linear estimate')
ax.plot(altitude_grid_km, tau_quadratic_grid, '--', color='tab:green', lw=2.5, label='tau quadratic estimate')
ax.scatter(target_altitudes_km, tau_avg_target, s=70, color='tab:orange', label='tau averaged values')

for altitude, tau_value in zip(target_altitudes_km, tau_avg_target):
    ax.annotate(
        f'{altitude:.0f} km: tau={tau_value:.2f} y',
        xy=(altitude, tau_value),
        xytext=(8, 8),
        textcoords='offset points',
        fontsize=9,
    )

ax.set_xlabel('Altitude [km]')
ax.set_ylabel('Perturbation lifetime tau [years]')
ax.set_title('Perturbation lifetime estimates by altitude')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### Compute factor `s`


In [ ]:
factor_s_target = approximate_factor_s(tau_avg_target)

summary = {
    float(altitude): {
        'tau_linear_years': round(float(tau_lin), 2),
        'tau_quadratic_years': round(float(tau_quad), 2),
        'tau_average_years': round(float(tau_avg), 2),
        'factor_s': round(float(s_val), 2),
    }
    for altitude, tau_lin, tau_quad, tau_avg, s_val in zip(
        target_altitudes_km,
        tau_linear_target,
        tau_quadratic_target,
        tau_avg_target,
        factor_s_target,
    )
}

summary


## 3. How to apply `s` in model runs

- Apply factor `s` to aircraft emissions only during the first simulation year.
- Reset emissions to original values after year 1.
- Apply the same approach consistently to affected trace-gas families (for example, `NO -> NOy`, `H2O/H2 -> HOx`) when the transport-timescale assumption is used.
- For emissions distributed over several altitudes, compute and apply altitude-specific `s` values.
